# MedGraph — Exportação para GGUF e publicação

**Tech Challenge Fase 3 · 8IADT**

Segundo notebook do Colab. Roda **depois** de `01_finetune_qlora_pubmedqa.ipynb`.

---

## O que este notebook faz

Transforma o adapter LoRA treinado em um modelo que roda offline num MacBook:

```
adapter LoRA (50 MB)
      │
      ▼  fusão com o modelo base
modelo completo em fp16 (~6 GB)
      │
      ▼  conversão + quantização (llama.cpp)
GGUF Q4_K_M (~2 GB)
      │
      ▼  publicação
Hugging Face Hub
      │
      ▼  make modelo -- --ajustado
Ollama, na máquina local
```

## Por que passar por GGUF em vez de rodar o adapter direto

Rodar `transformers` + `peft` no Apple Silicon funciona, mas fica em torno de
8 tokens/s num modelo de 3B — inviável para uma demonstração interativa e para o
vídeo de entrega. O mesmo modelo em GGUF `Q4_K_M` servido pelo Ollama passa de
30 tokens/s no mesmo hardware, usando Metal.

Há um segundo motivo, mais importante para o projeto: a **Aula 05** do curso usou
Ollama. Servir o modelo por ele mantém o código LangChain idêntico ao padrão
ensinado — muda apenas o nome do modelo, que agora é o nosso.

## Por que `Q4_K_M`

| Quantização | Tamanho | Qualidade | Cabe em 16 GB de RAM |
|---|---|---|---|
| F16 | ~6,4 GB | referência | sim, apertado |
| Q8_0 | ~3,4 GB | quase idêntica | sim |
| **Q4_K_M** | **~2,0 GB** | **perda pequena** | **sim, com folga** |
| Q4_0 | ~1,9 GB | perda perceptível | sim |

`Q4_K_M` é o ponto de equilíbrio usual: usa 6 bits nas camadas mais sensíveis
(atenção `v` e `feed-forward down`) e 4 bits no resto.

## 1. Ambiente

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Dependências e repositório

In [ ]:
%pip install -q -U "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" "huggingface_hub>=0.26" sentencepiece protobuf

import os, sys
if not os.path.isdir("/content/fia_tech3"):
    !git clone --depth 1 https://github.com/alexandreccarmo/fia_tech3.git /content/fia_tech3
os.chdir("/content/fia_tech3")
sys.path.insert(0, "/content/fia_tech3/src")
sys.path.insert(0, "/content/fia_tech3")

## 3. O adapter

Duas origens possíveis:

- **A** — o notebook 01 acabou de rodar nesta mesma sessão: o adapter já está em disco.
- **B** — sessão nova: faça o upload do `medgraph-adapter.zip` baixado no notebook 01.

In [ ]:
from pathlib import Path

ADAPTER = Path("models/adapters/medgraph-llama32-3b-lora")

if not (ADAPTER / "adapter_config.json").exists():
    print("Adapter não encontrado. Faça o upload de medgraph-adapter.zip:")
    from google.colab import files
    enviados = files.upload()
    for nome in enviados:
        !unzip -qo {nome} -d models/adapters/

assert (ADAPTER / "adapter_config.json").exists(), "adapter ainda ausente"
!ls -la {ADAPTER}

import json
cartao = ADAPTER / "cartao_de_treino.json"
if cartao.exists():
    dados = json.loads(cartao.read_text())
    print("\nCartão de treino")
    print(f"  modelo base ......... {dados['modelo_base']}")
    print(f"  exemplos de treino .. {dados['dados']['treino']:,}")
    print(f"  duração ............. {dados['duracao_legivel']}")
    print(f"  perda de validação .. {dados['perda_validacao_inicial']} -> {dados['perda_validacao_final']}")

## 4. Autenticação

In [ ]:
from huggingface_hub import login, whoami
login()
USUARIO = whoami()["name"]
print("Autenticado como:", USUARIO)

## 5. Fusão do adapter ao modelo base

A fusão soma as matrizes de baixo posto aos pesos originais, produzindo um modelo
comum — sem dependência de `peft` na hora de usar.

O modelo base é recarregado em **fp16, não em 4 bits**. Fundir sobre pesos
quantizados propagaria o erro de quantização para dentro dos pesos finais, e a
perda se somaria à da quantização seguinte, para GGUF. Fundindo em fp16,
quantizamos uma vez só.

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODELO_BASE = "meta-llama/Llama-3.2-3B-Instruct"
FUNDIDO = "/content/medgraph-fundido"

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizador.pad_token is None:
    tokenizador.pad_token = tokenizador.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE, dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True
)
modelo = PeftModel.from_pretrained(base, str(ADAPTER))
modelo = modelo.merge_and_unload()

modelo.save_pretrained(FUNDIDO, safe_serialization=True)
tokenizador.save_pretrained(FUNDIDO)

del modelo, base
gc.collect(); torch.cuda.empty_cache()

!du -sh {FUNDIDO}

## 6. llama.cpp

A compilação leva alguns minutos. Só precisamos do utilitário de quantização e do script de conversão.

In [ ]:
import os
if not os.path.isdir("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp

%pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null 2>&1

!find /content/llama.cpp/build -name "llama-quantize" -type f

## 7. Conversão para GGUF e quantização

In [ ]:
GGUF_F16 = "/content/medgraph-llama32-3b-f16.gguf"
GGUF_Q4  = "/content/medgraph-llama32-3b-q4_k_m.gguf"

!python /content/llama.cpp/convert_hf_to_gguf.py {FUNDIDO} --outfile {GGUF_F16} --outtype f16

import glob
quantizador = glob.glob("/content/llama.cpp/build/**/llama-quantize", recursive=True)[0]
!{quantizador} {GGUF_F16} {GGUF_Q4} Q4_K_M

!ls -lh /content/*.gguf

## 8. Publicação no Hugging Face Hub

O GGUF tem cerca de 2 GB — grande demais para o repositório Git. Fica no Hub, e o
comando `make modelo -- --ajustado` o recupera na máquina local.

O `Modelfile` também é enviado junto, para que qualquer pessoa consiga registrar o
modelo no Ollama a partir do Hub.

In [ ]:
from huggingface_hub import HfApi, create_repo

REPO_ID = f"{USUARIO}/medgraph-llama32-3b-gguf"
create_repo(REPO_ID, repo_type="model", exist_ok=True)

api = HfApi()
api.upload_file(
    path_or_fileobj=GGUF_Q4,
    path_in_repo="medgraph-llama32-3b-q4_k_m.gguf",
    repo_id=REPO_ID,
)
api.upload_file(
    path_or_fileobj="src/medgraph/finetune/Modelfile",
    path_in_repo="Modelfile",
    repo_id=REPO_ID,
)

print(f"Publicado em https://huggingface.co/{REPO_ID}")
print("\nAtualize o .env local com:")
print(f"  REPO_GGUF_HF={REPO_ID}")

## 9. Alternativa — download direto

Caso prefira não publicar no Hub. São ~2 GB pelo navegador, então costuma ser mais
lento e sujeito a interrupção.

In [ ]:
# from google.colab import files
# files.download(GGUF_Q4)

---

## Próximos passos, na máquina local

```bash
# 1. Ajuste o .env com o repositório publicado
#    REPO_GGUF_HF=seu-usuario/medgraph-llama32-3b-gguf

# 2. Baixe o GGUF e registre no Ollama
make modelo

# 3. Confirme
ollama list
ollama run medgraph "Qual a conduta inicial na suspeita de sepse?"

# 4. Avalie o modelo
make avaliar
```

## Se algo der errado

| Sintoma | Causa | Solução |
|---|---|---|
| Falta de RAM na fusão | Colab gratuito tem ~12 GB | Ambiente de execução → alta RAM, ou use Qwen2.5-1.5B |
| `convert_hf_to_gguf.py` não reconhece a arquitetura | llama.cpp desatualizado | remova `/content/llama.cpp` e clone de novo |
| `llama-quantize` não encontrado | falha na compilação | rode o `cmake --build` sem `> /dev/null` e leia o erro |
| Upload interrompido | arquivo de 2 GB | reexecute a célula: o Hub retoma de onde parou |